# AI-Powered Technical Debt Quantification and Remediation
## API Notebook: Tool Exploration and Building Blocks

**Course:** DATA605 — Big Data Systems, Spring 2026  
**Team:** Akhil Kambhatla, Namratha Jeetendra, Hemanth Thulasiraman  
**Dataset:** The Technical Debt Dataset V2 (Lenarduzzi et al., 2019)

---

### What Is Technical Debt?

Technical debt is a concept from software engineering. When developers take shortcuts to ship code faster, they create problems that need to be fixed later. These shortcuts might be overly complex functions, duplicated code blocks, missing tests, or known bugs left unfixed. Just like financial debt, technical debt accumulates interest: the longer you wait to fix it, the more expensive it becomes.

SonarQube, one of the most widely used code analysis tools, classifies technical debt into three types:
- **Bugs:** Code that is objectively wrong and will likely cause failures
- **Code Smells:** Code that works but is hard to maintain, read, or extend
- **Vulnerabilities:** Code with security weaknesses

### Why Does This Matter?

According to the CISQ 2022 report, accumulated software technical debt in the US has grown to approximately $1.52 trillion (CISQ, 2022).

Recent work by Tornhill et al. (FSE 2025) introduced **ACE** (Augmented Code Engineering), a tool that uses LLMs to automatically refactor code and reduce technical debt. Their benchmarking study on over 100,000 real-world code issues found that the best-performing LLM generated functionally correct refactorings only 37% of the time when used without any validation. By adding a metric-driven validation pipeline that filters out bad suggestions, ACE achieved 98% precision on the remaining refactorings, though at the cost of only keeping 52% of the suggestions (the rest were discarded as unreliable).

However, ACE relies on proprietary tools (CodeScene, CodeHealth metric) and commercial LLMs. Our project investigates whether the same grounded approach works with fully open-source tools and a local code model that anyone can run for free.

### Our Research Question

**Can a fully open-source pipeline that combines ML-based fault prediction with local LLM code generation achieve effective technical debt remediation without commercial tools or API access?**

### What This Notebook Covers

This API notebook demonstrates each tool individually:

1. **Code metrics** with `radon` — measuring cyclomatic complexity
2. **Advanced metrics** with `lizard` — multi-language analysis
3. **Code smells** with `pylint` — detecting style and design violations
4. **AST analysis** with Python's `ast` module — understanding code structure
5. **Git mining** with `PyDriller` — extracting commit history
6. **Querying the Technical Debt Dataset** — SQL on real-world data
7. **Debt type classification** — training a classifier with scikit-learn
8. **Impact prediction** — training a regressor with XGBoost
9. **Code generation** — loading a pretrained code model
10. **Agent building blocks** — LangGraph tool-use patterns

The **Example notebook** combines all of these into an end-to-end pipeline and compares three approaches: SonarQube defaults, direct LLM prompting, and our grounded pipeline.

### Setup

All utility functions are defined in `ai_technical_debt_utils.py`. This notebook imports from that module and demonstrates each component individually.

In [1]:
# Standard imports.
import sys
import os
import warnings

# Add the project directory to the path so we can import our utils.
sys.path.insert(0, "/curr_dir")

# Suppress noisy warnings for cleaner notebook output.
warnings.filterwarnings("ignore")
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

# Project imports.
from ai_technical_debt_utils import *

print("All imports successful.")

All imports successful.


---
## Section 1: Code Complexity with `radon`

`radon` is a Python tool that computes code metrics, including:
- **Cyclomatic Complexity (CC):** the number of independent execution paths through a function. Higher CC means more branches, more test cases needed, and more cognitive load for developers. This is the same metric SonarQube computes for Java code in our Technical Debt Dataset.
- **Halstead Metrics:** measures of code "size" based on counting operators and operands.
- **Maintainability Index (MI):** a composite score (0-100) combining complexity, code length, and Halstead volume. Higher is better.

We use `radon` on Python code here to teach the concept. Later, in Section 6, we query the same metrics precomputed at scale on Java projects.

In [2]:
# We'll analyze two versions of the same function: one clean, one messy.
# This shows how complexity changes with coding style.

clean_code = '''
def calculate_grade(score):
    """Convert a numeric score to a letter grade."""
    if score >= 90:
        return "A"
    elif score >= 80:
        return "B"
    elif score >= 70:
        return "C"
    elif score >= 60:
        return "D"
    else:
        return "F"
'''

messy_code = '''
def process_student_data(students, include_inactive=False, sort_by=None,
                         min_score=0, max_score=100, normalize=False):
    """Process student records with multiple filtering and sorting options."""
    results = []
    for student in students:
        if not include_inactive and not student.get("active", True):
            continue
        score = student.get("score", 0)
        if score < min_score or score > max_score:
            continue
        if normalize and max_score != min_score:
            score = (score - min_score) / (max_score - min_score) * 100
        if score >= 90:
            grade = "A"
        elif score >= 80:
            grade = "B"
        elif score >= 70:
            grade = "C"
        elif score >= 60:
            grade = "D"
        else:
            grade = "F"
        result = {"name": student.get("name", "Unknown"), "grade": grade}
        if sort_by and sort_by in student:
            result["sort_key"] = student[sort_by]
        results.append(result)
    if sort_by:
        try:
            results.sort(key=lambda x: x.get("sort_key", ""))
        except TypeError:
            pass
    return results
'''

print("Code samples loaded.")
print(f"Clean function: {len(clean_code.strip().splitlines())} lines")
print(f"Messy function: {len(messy_code.strip().splitlines())} lines")

Code samples loaded.
Clean function: 12 lines
Messy function: 32 lines


In [3]:
# Compute cyclomatic complexity for both functions.
from radon.complexity import cc_visit, cc_rank

clean_results = cc_visit(clean_code)
messy_results = cc_visit(messy_code)

print("=== Clean Function ===")
for item in clean_results:
    print(f"  Function: {item.name}")
    print(f"  Cyclomatic Complexity: {item.complexity}")
    print(f"  Grade: {cc_rank(item.complexity)}")

print("\n=== Messy Function ===")
for item in messy_results:
    print(f"  Function: {item.name}")
    print(f"  Cyclomatic Complexity: {item.complexity}")
    print(f"  Grade: {cc_rank(item.complexity)}")

=== Clean Function ===
  Function: calculate_grade
  Cyclomatic Complexity: 5
  Grade: A

=== Messy Function ===
  Function: process_student_data
  Cyclomatic Complexity: 16
  Grade: C


Radon classifies complexity using letter grades:
- **A (1-5):** Low risk, simple block
- **B (6-10):** Low risk, well-structured and stable block
- **C (11-20):** Moderate, slightly complex block
- **D (21-30):** More than moderate, more complex block
- **E (31-40):** High, complex block, alarming
- **F (41+):** Very high, error-prone, unstable block

The clean function should score A (simple), while the messy function should score C (moderately complex) with its 16 branches and nested conditions.

In [4]:
# Compute Maintainability Index for both.
from radon.metrics import mi_visit

clean_mi = mi_visit(clean_code, multi=False)
messy_mi = mi_visit(messy_code, multi=False)

print("=== Maintainability Index (0-100, higher is better) ===")
print(f"  Clean function: {clean_mi:.1f}")
print(f"  Messy function: {messy_mi:.1f}")

if clean_mi > messy_mi:
    print(f"\n  The clean version is {clean_mi - messy_mi:.1f} points more maintainable.")

=== Maintainability Index (0-100, higher is better) ===
  Clean function: 65.3
  Messy function: 47.8

  The clean version is 17.6 points more maintainable.


---
## Section 2: Multi-Language Metrics with `lizard`

While `radon` only works on Python, `lizard` supports over 20 languages including Python, Java, C, C++, and JavaScript. This is relevant because the Technical Debt Dataset contains Java projects analyzed by SonarQube.

For each function, `lizard` computes:
- **NLOC:** Lines of code without comments
- **CCN:** Cyclomatic complexity number (same concept as radon's CC)
- **Token count:** Total number of code tokens (operators, identifiers, literals)
- **Parameter count:** Number of function parameters

We can also analyze Java code directly, which is useful since our dataset is entirely Java.

In [5]:
import lizard

# Analyze our Python functions using lizard.
# analyze_source_code requires a filename to identify the language.
clean_analysis = lizard.analyze_file.analyze_source_code(
    "clean.py", clean_code
)
messy_analysis = lizard.analyze_file.analyze_source_code(
    "messy.py", messy_code
)

print("=== Clean Function (Python) ===")
for func in clean_analysis.function_list:
    print(f"  Function: {func.name}")
    print(f"  NLOC: {func.nloc}")
    print(f"  CCN: {func.cyclomatic_complexity}")
    print(f"  Tokens: {func.token_count}")
    print(f"  Parameters: {func.parameter_count}")

print("\n=== Messy Function (Python) ===")
for func in messy_analysis.function_list:
    print(f"  Function: {func.name}")
    print(f"  NLOC: {func.nloc}")
    print(f"  CCN: {func.cyclomatic_complexity}")
    print(f"  Tokens: {func.token_count}")
    print(f"  Parameters: {func.parameter_count}")

=== Clean Function (Python) ===
  Function: calculate_grade
  NLOC: 11
  CCN: 5
  Tokens: 38
  Parameters: 1

=== Messy Function (Python) ===
  Function: process_student_data
  NLOC: 31
  CCN: 16
  Tokens: 198
  Parameters: 6


In [6]:
# lizard can also analyze Java code directly.
# This is relevant because our Technical Debt Dataset contains Java projects.
java_code = '''
public class Calculator {
    public double calculate(String operation, double a, double b) {
        if (operation.equals("add")) {
            return a + b;
        } else if (operation.equals("subtract")) {
            return a - b;
        } else if (operation.equals("multiply")) {
            return a * b;
        } else if (operation.equals("divide")) {
            if (b == 0) {
                throw new ArithmeticException("Division by zero");
            }
            return a / b;
        } else {
            throw new IllegalArgumentException("Unknown operation: " + operation);
        }
    }
}
'''

java_analysis = lizard.analyze_file.analyze_source_code(
    "Calculator.java", java_code
)

print("=== Java Function ===")
for func in java_analysis.function_list:
    print(f"  Function: {func.name}")
    print(f"  NLOC: {func.nloc}")
    print(f"  CCN: {func.cyclomatic_complexity}")
    print(f"  Tokens: {func.token_count}")
    print(f"  Parameters: {func.parameter_count}")

=== Java Function ===
  Function: Calculator::calculate
  NLOC: 16
  CCN: 6
  Tokens: 107
  Parameters: 3


Notice that `lizard` produces the same cyclomatic complexity as `radon` for the Python functions, but can also analyze Java code without any additional setup. This multi-language capability is what makes tools like `lizard` and SonarQube valuable for large organizations with mixed-language codebases.

In our Technical Debt Dataset, SonarQube computed these same metrics (complexity, NLOC, etc.) for all 154,000 commits across 31 Apache Java projects. The `SONAR_MEASURES` table contains these values precomputed at scale.

---
## Section 3: Code Smells with `pylint`

`radon` and `lizard` measure complexity (how many paths through the code). `pylint` is different: it checks whether the code follows good practices and coding standards. It detects things like:

- Unused variables and imports
- Missing docstrings
- Variable names that are too short or don't follow conventions
- Functions with too many arguments or local variables
- Duplicate code patterns

`pylint` scores code from 0 to 10, where 10 means fully compliant with coding standards. In the Technical Debt Dataset, SonarQube performs a similar role: it detects "code smells" (maintainability issues) and "bugs" (correctness issues) using its own rule set of over 500 rules for Java.

In [7]:
import tempfile
import os
from pylint.lint import Run
from pylint.reporters.text import TextReporter
import io

# Write the messy code to a temporary file for pylint to analyze.
with tempfile.NamedTemporaryFile(
    mode="w", suffix=".py", delete=False
) as f:
    f.write(messy_code)
    temp_path = f.name

# Run pylint and capture the output.
output = io.StringIO()
reporter = TextReporter(output)
try:
    Run(
        [temp_path, "--disable=C0114,C0115,C0116"],  # Disable module/class docstring warnings
        reporter=reporter,
        exit=False,
    )
except SystemExit:
    pass

# Print the results.
pylint_output = output.getvalue()
print(pylint_output)

# Clean up.
os.unlink(temp_path)

************* Module tmpojqp8009
/tmp/tmpojqp8009.py:2:0: R0913: Too many arguments (6/5) (too-many-arguments)
/tmp/tmpojqp8009.py:2:0: R0917: Too many positional arguments (6/5) (too-many-positional-arguments)

-----------------------------------
Your code has been rated at 9.31/10




Notice something interesting: pylint gave the messy function 9.31/10 despite it having a cyclomatic complexity of 16 (grade C in radon). This is because **pylint and radon measure different things**. Pylint checks coding conventions and style. Radon measures structural complexity. Code can be perfectly formatted but deeply complex, or badly formatted but simple.

This is why our pipeline uses multiple metrics rather than relying on a single tool. In the Technical Debt Dataset, SonarQube combines both approaches: it has convention rules (similar to pylint) and complexity rules (similar to radon) in a single analysis.

The two issues pylint did find (`R0913: Too many arguments` and `R0917: Too many positional arguments`) are both in the **R (Refactor)** category, meaning pylint recommends restructuring the function. These correspond directly to what SonarQube would classify as a `CODE_SMELL`.

Each pylint message has a code like `C0301` or `R0913`. The first letter tells you the category:

- **C (Convention):** Style violations (naming, line length, formatting)
- **R (Refactor):** Code that should be restructured (too many arguments, too many branches)
- **W (Warning):** Potential problems that might cause bugs
- **E (Error):** Actual errors in the code
- **F (Fatal):** Errors that prevented pylint from analyzing the code

The **R** category (Refactor) is most relevant to technical debt: these are signals that code needs structural improvement. Messages like `R0913: Too many arguments` or `R0912: Too many branches` directly correspond to the kinds of issues SonarQube flags as code smells in our dataset.

---
## Section 4: Code Structure with Python's `ast` Module

The `ast` (Abstract Syntax Tree) module is built into Python. It parses source code into a tree structure that represents the code's logical organization: which functions exist, how deeply nested they are, what operations they perform.

Unlike radon, lizard, and pylint (which are third-party tools), `ast` gives you raw access to the code's structure. This is useful for building custom metrics that standard tools don't provide, such as:

- Counting the number of functions and classes in a file
- Measuring maximum nesting depth (how many levels of indentation)
- Detecting function calls to specific libraries
- Finding hardcoded values that should be constants

In [8]:
import ast

# Parse the messy code into an AST.
tree = ast.parse(messy_code)

# Walk the tree and count different node types.
node_counts = {}
for node in ast.walk(tree):
    name = type(node).__name__
    node_counts[name] = node_counts.get(name, 0) + 1

# Show the most common node types.
print("=== AST Node Counts (top 10) ===")
sorted_counts = sorted(node_counts.items(), key=lambda x: -x[1])
for name, count in sorted_counts[:10]:
    print(f"  {name}: {count}")

=== AST Node Counts (top 10) ===
  Name: 44
  Load: 42
  Constant: 27
  Store: 11
  Assign: 10
  If: 9
  Compare: 8
  arg: 7
  Call: 6
  Attribute: 6


In [9]:
def analyze_code_structure(source_code):
    """
    Extract structural metrics from Python source code using the ast module.
    
    Returns a dictionary with:
    - num_functions: number of function definitions
    - num_classes: number of class definitions
    - max_depth: maximum nesting depth of control structures
    - num_branches: total number of if/elif/else branches
    - num_loops: total number of for/while loops
    - num_try_except: total number of try/except blocks
    """
    tree = ast.parse(source_code)
    
    metrics = {
        "num_functions": 0,
        "num_classes": 0,
        "num_branches": 0,
        "num_loops": 0,
        "num_try_except": 0,
    }
    
    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef):
            metrics["num_functions"] += 1
        elif isinstance(node, ast.ClassDef):
            metrics["num_classes"] += 1
        elif isinstance(node, (ast.If,)):
            metrics["num_branches"] += 1
        elif isinstance(node, (ast.For, ast.While)):
            metrics["num_loops"] += 1
        elif isinstance(node, ast.Try):
            metrics["num_try_except"] += 1
    
    return metrics

print("=== Clean Function ===")
clean_metrics = analyze_code_structure(clean_code)
for key, value in clean_metrics.items():
    print(f"  {key}: {value}")

print("\n=== Messy Function ===")
messy_metrics = analyze_code_structure(messy_code)
for key, value in messy_metrics.items():
    print(f"  {key}: {value}")

=== Clean Function ===
  num_functions: 1
  num_classes: 0
  num_branches: 4
  num_loops: 0
  num_try_except: 0

=== Messy Function ===
  num_functions: 1
  num_classes: 0
  num_branches: 9
  num_loops: 1
  num_try_except: 1


The AST analysis confirms what radon and lizard told us: the messy function has more branches, more loops, and a try/except block that the clean function lacks. 

Custom AST analysis like this is valuable when you need metrics that standard tools don't provide. For example, you could write an AST visitor that counts how many external library calls a function makes (a measure of coupling), or that detects deeply nested loops (a performance concern).

In our pipeline, we use precomputed metrics from SonarQube rather than building custom AST analyzers. But understanding how AST analysis works helps explain what tools like SonarQube are doing under the hood.

---
## Section 5: Git Mining with `PyDriller`

`PyDriller` is the tool the Technical Debt Dataset authors used to extract commit history from 33 Apache Java projects (Lenarduzzi et al., 2019). It walks through a Git repository's history and extracts structured data about each commit: who made it, when, what files changed, how many lines were added or removed, and the actual diffs.

This is how the `GIT_COMMITS` and `GIT_COMMITS_CHANGES` tables in our dataset were populated. Here we demonstrate PyDriller on a live repository to show how it works.

In [10]:
# Git inside Docker needs to trust the mounted repository directory.
# Without this, PyDriller fails on the first run because Git refuses
# to operate on a directory owned by a different user.
import subprocess
subprocess.run(
    ["git", "config", "--global", "--add", "safe.directory", "/git_root"],
    capture_output=True,
)
print("Git safe directory configured.")

Git safe directory configured.


In [11]:
from pydriller import Repository

# Mine the last 5 commits from this project's own repository.
# The repo is mounted at /git_root inside Docker.
repo_path = "/git_root"

commits = []
for commit in Repository(repo_path, order="reverse").traverse_commits():
    commits.append({
        "hash": commit.hash[:8],
        "author": commit.author.name,
        "date": commit.author_date.strftime("%Y-%m-%d %H:%M"),
        "message": commit.msg.split("\n")[0][:80],
        "files_changed": commit.files,
        "insertions": commit.insertions,
        "deletions": commit.deletions,
    })
    if len(commits) >= 5:
        break

import pandas as pd
commits_df = pd.DataFrame(commits)
print(f"Showing the 5 most recent commits from {repo_path}:\n")
commits_df

INFO:pydriller.repository:Analyzing git repository in /git_root
INFO:pydriller.repository:Commit #4b6e9fe082fc27a93e77de9546f60161fc280708 in 2026-04-18 14:19:46-04:00 from Akhil
INFO:pydriller.repository:Commit #af122b8ebe9c9513e7ddfe2ca50faa5550513cfe in 2026-04-18 12:40:15-04:00 from Akhil
INFO:pydriller.repository:Commit #ddcebdaa16655eedd5e12597445c84521148663d in 2026-04-18 12:19:26-04:00 from Akhil
INFO:pydriller.repository:Commit #8867ba6fcea775947dd23d024c7c5368a8a742ad in 2026-04-17 16:27:43-04:00 from Akhil
INFO:pydriller.repository:Commit #fb2c04ca0b6c2a330cca597f0d12852dc2523e7c in 2026-04-16 21:55:31-04:00 from Akhil


Showing the 5 most recent commits from /git_root:



,hash,author,date,message,files_changed,insertions,deletions
0,4b6e9fe0,Akhil,2026-04-18 14:19,"[API Notebook] Add reproducibility settings, f...",1,556,714
1,af122b8e,Akhil,2026-04-18 12:40,[API Notebook] Add sections 7-8: fault predict...,1,549,65
2,ddcebdaa,Akhil,2026-04-18 12:19,"[API Notebook] Add sections 0-6: intro, radon,...",1,1582,84
3,8867ba6f,Akhil,2026-04-17 16:27,[Module 1] Add data loading and feature engine...,2,583,44
4,fb2c04ca,Akhil,2026-04-16 21:55,"[Setup] Rename template files, configure Docke...",9,156,37


In [12]:
# Look at the details of one commit: which files changed and how.
for commit in Repository(repo_path, order="reverse").traverse_commits():
    if commit.files > 0:
        print(f"Commit: {commit.hash[:8]}")
        print(f"Author: {commit.author.name}")
        print(f"Message: {commit.msg.split(chr(10))[0]}")
        print(f"\nFiles modified:")
        for mod_file in commit.modified_files:
            print(f"  {mod_file.filename}")
            print(f"    Lines added: {mod_file.added_lines}")
            print(f"    Lines removed: {mod_file.deleted_lines}")
            print(f"    Change type: {mod_file.change_type.name}")
            if mod_file.complexity is not None:
                print(f"    Complexity: {mod_file.complexity}")
        break  # Only show one commit.

INFO:pydriller.repository:Analyzing git repository in /git_root
INFO:pydriller.repository:Commit #4b6e9fe082fc27a93e77de9546f60161fc280708 in 2026-04-18 14:19:46-04:00 from Akhil


Commit: 4b6e9fe0
Author: Akhil
Message: [API Notebook] Add reproducibility settings, finalize all markdown explanations

Files modified:
  ai_technical_debt.API.ipynb
    Lines added: 556
    Lines removed: 714
    Change type: MODIFY


PyDriller gives us structured access to the same data that populates the `GIT_COMMITS` and `GIT_COMMITS_CHANGES` tables in the Technical Debt Dataset. The key fields are:

- **Files changed, insertions, deletions:** Measures the size of each commit
- **Change type:** Whether a file was Added, Modified, Deleted, or Renamed
- **Complexity:** PyDriller can compute cyclomatic complexity per file using `lizard` internally

In the Technical Debt Dataset, this information was collected for all 153,994 commits across 31 projects. In the next section, we query that precomputed data directly.

---
## Section 6: Querying the Technical Debt Dataset

In Sections 1-5, we demonstrated individual tools on small code samples. Now we connect to the actual dataset used in our research: the Technical Debt Dataset V2 (Lenarduzzi et al., 2019).

This dataset contains the output of running SonarQube, Ptidej, Refactoring Miner, and the SZZ algorithm across 31 Apache Java projects. The data is stored as a SQLite database with 10 tables covering:

- **Code metrics** (complexity, coverage, duplication) per commit
- **Detected issues** (bugs, code smells, vulnerabilities) with severity and remediation effort
- **Refactoring operations** applied by developers
- **Fault-inducing commits** identified by the SZZ algorithm
- **Jira issue reports** from the project tracker

All data loading functions are defined in `ai_technical_debt_utils.py`.

In [13]:
# Connect to the database and get an overview.
conn = connect_to_database("/curr_dir/data/td_V2.db")
summary = get_dataset_summary(conn)
summary

INFO:ai_technical_debt_utils:Connecting to database: /curr_dir/data/td_V2.db
INFO:ai_technical_debt_utils:Dataset summary:
                TABLE_NAME  ROW_COUNT
               GIT_COMMITS     153994
       GIT_COMMITS_CHANGES    1142878
            SONAR_MEASURES      66711
              SONAR_ISSUES    1024614
            SONAR_ANALYSIS      67550
               SONAR_RULES       1819
         REFACTORING_MINER     362253
               JIRA_ISSUES      61402
SZZ_FAULT_INDUCING_COMMITS      52428
                  PROJECTS         31


,TABLE_NAME,ROW_COUNT
0,GIT_COMMITS,153994
1,GIT_COMMITS_CHANGES,1142878
2,SONAR_MEASURES,66711
3,SONAR_ISSUES,1024614
4,SONAR_ANALYSIS,67550
5,SONAR_RULES,1819
6,REFACTORING_MINER,362253
7,JIRA_ISSUES,61402
8,SZZ_FAULT_INDUCING_COMMITS,52428
9,PROJECTS,31


In [14]:
# List all projects in the dataset.
projects = get_projects(conn)
print(f"Total projects: {len(projects)}\n")
projects[["PROJECT_ID", "PROJECT_KEY"]]

INFO:ai_technical_debt_utils:Loaded 31 projects


Total projects: 31



,PROJECT_ID,PROJECT_KEY
0,org.apache:batik,batik
1,org.apache:bcel,commons-bcel
2,org.apache:beanutils,commons-beanutils
3,org.apache:cocoon,cocoon
4,org.apache:codec,commons-codec
5,org.apache:collections,commons-collections
6,org.apache:commons-cli,commons-cli
7,org.apache:commons-exec,commons-exec
8,org.apache:commons-fileupload,commons-fileupload
9,org.apache:commons-io,commons-io


### Exploring Code Metrics

The `SONAR_MEASURES` table contains 30+ code metrics per commit snapshot. Let's look at one project to understand what the data looks like.

In [15]:
# Load metrics for a medium-sized project.
measures = get_sonar_measures(conn, "org.apache:commons-io")
print(f"Shape: {measures.shape}")
print(f"\nNumeric columns available:")

# Show a subset of the most important metrics.
key_metrics = [
    "COMPLEXITY", "COGNITIVE_COMPLEXITY", "COVERAGE",
    "DUPLICATED_LINES_DENSITY", "NCLOC", "FUNCTIONS",
    "CLASSES",
]
available = [c for c in key_metrics if c in measures.columns]
measures[available].describe().round(2)

INFO:ai_technical_debt_utils:Loaded 1910 SONAR_MEASURES rows for project org.apache:commons-io


Shape: (1910, 240)

Numeric columns available:


,COMPLEXITY,COGNITIVE_COMPLEXITY,COVERAGE,NCLOC,FUNCTIONS,CLASSES
count,1908.00,1908.00,1908.0,1908.00,1908.00,1908.00
mean,2654.92,1725.21,0.0,17065.10,1643.60,164.10
std,1232.13,837.56,0.0,8010.86,754.55,71.39
min,55.00,31.00,0.0,312.00,34.00,4.00
25%,1470.00,926.00,0.0,9489.00,907.00,89.00
50%,2781.00,1808.50,0.0,18162.00,1777.00,190.00
75%,3797.00,2519.00,0.0,24194.25,2358.00,228.00
max,4491.00,2858.00,0.0,32945.00,2702.00,272.00


### Exploring Technical Debt Issues

The `SONAR_ISSUES` table contains over 1 million individual issues detected by SonarQube and Ptidej. Each issue has a type, severity, and estimated remediation effort.

In [16]:
# Load issues for the same project.
issues = get_sonar_issues(conn, "org.apache:commons-io")
print(f"Total issues for commons-io: {len(issues)}\n")

# Distribution by type.
print("Issues by TYPE:")
print(issues["TYPE"].value_counts().to_string())

print("\n\nIssues by SEVERITY:")
print(issues["SEVERITY"].value_counts().to_string())

INFO:ai_technical_debt_utils:Loaded 4000 SONAR_ISSUES rows for project org.apache:commons-io


Total issues for commons-io: 4000

Issues by TYPE:
TYPE
CODE_SMELL       3888
BUG                96
VULNERABILITY      16


Issues by SEVERITY:
SEVERITY
MAJOR       2001
MINOR       1198
CRITICAL     589
INFO         200
BLOCKER       12


### Fault-Inducing Commits and the SZZ Algorithm

The `SZZ_FAULT_INDUCING_COMMITS` table links Jira bug reports to the specific commits that introduced them. This was computed using the SZZ algorithm, which traces bug-fixing commits back through Git history to find the original fault-inducing commit.

This data is the ground truth for our impact prediction model: we want to predict which commits will turn out to be fault-inducing based on their code metrics at the time of the commit.

In [17]:
# Load fault-inducing commit data.
faults = get_fault_inducing_commits(conn, "org.apache:commons-io")
print(f"Fault-inducing commit records: {len(faults)}")
print(f"Unique fault-inducing commits: {faults['FAULT_INDUCING_COMMIT_HASH'].nunique()}")
print(f"Unique fault-fixing commits: {faults['FAULT_FIXING_COMMIT_HASH'].nunique()}")

INFO:ai_technical_debt_utils:Loaded 1452 fault-inducing commit records for project org.apache:commons-io


Fault-inducing commit records: 1452
Unique fault-inducing commits: 597
Unique fault-fixing commits: 126


### Building the Feature Matrix

Our `build_full_feature_matrix()` function joins the metrics, issues, and fault labels into a single dataframe ready for machine learning. Each row is one commit with all its metrics as features and a binary label indicating whether it introduced a fault.

In [18]:
# Build the full feature matrix for commons-io.
feature_matrix = build_full_feature_matrix(conn, "org.apache:commons-io")
print(f"Feature matrix shape: {feature_matrix.shape}")
print(f"Fault-inducing commits: {feature_matrix['IS_FAULT_INDUCING'].sum()} "
      f"/ {len(feature_matrix)} "
      f"({100 * feature_matrix['IS_FAULT_INDUCING'].mean():.1f}%)")

# Show the class balance.
print(f"\nClass distribution:")
print(feature_matrix["IS_FAULT_INDUCING"].value_counts().to_string())

INFO:ai_technical_debt_utils:Project org.apache:commons-io: 1910 commits with metrics
INFO:ai_technical_debt_utils:Loaded 1452 fault-inducing commit records for project org.apache:commons-io
INFO:ai_technical_debt_utils:Project org.apache:commons-io: 597 unique fault-inducing commits
INFO:ai_technical_debt_utils:Project org.apache:commons-io: 484 / 1910 commits are fault-inducing (25.3%)
INFO:ai_technical_debt_utils:Project org.apache:commons-io: issue counts for 402 commits
INFO:ai_technical_debt_utils:Project org.apache:commons-io: full feature matrix has 1910 rows and 252 columns


Feature matrix shape: (1910, 252)
Fault-inducing commits: 484 / 1910 (25.3%)

Class distribution:
IS_FAULT_INDUCING
0    1426
1     484


**Note:** All the queries above are for a single project (`commons-io`) as a demonstration. The full dataset contains 31 Apache projects with a combined 153,994 commits, 1,024,614 issues, and 52,428 fault-inducing commit records. In the Example notebook, we build feature matrices across multiple projects for training and evaluation.

---
## Section 7: Predicting Fault-Inducing Commits

Can we predict which commits will introduce bugs based on their code metrics?

This is the core of our impact quantification module (Module 3). We train a classifier on the feature matrix built in Section 6, where each row is one commit with SonarQube metrics as features and the SZZ fault-inducing label as the target.

We train two models:
- **Logistic Regression:** A simple, interpretable baseline
- **XGBoost:** A gradient-boosted tree model that can capture non-linear relationships

We evaluate with **F1 score** and **AUC (Area Under the ROC Curve)** rather than raw accuracy, because the data is imbalanced: most commits are not fault-inducing. A model that always predicts "safe" would get high accuracy but would be useless.

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, roc_auc_score, f1_score
)
from xgboost import XGBClassifier

# Use the feature matrix we already built for commons-io.
target = "IS_FAULT_INDUCING"

# Drop non-feature columns (IDs, hashes, dates, text).
drop_cols = [
    target, "PROJECT_ID", "ANALYSIS_KEY", "COMMIT_HASH",
    "ANALYSIS_DATE",
]
text_cols = [
    c for c in feature_matrix.columns
    if feature_matrix[c].dtype == "object"
]
drop_cols = [c for c in drop_cols + text_cols if c in feature_matrix.columns]

X = feature_matrix.drop(columns=drop_cols)
y = feature_matrix[target]

# Convert all remaining columns to numeric.
# Some columns in the database are stored as TEXT and may contain
# empty strings. pd.to_numeric with errors='coerce' turns those
# into NaN, which we then fill with 0.
X = X.apply(pd.to_numeric, errors="coerce").fillna(0)

print(f"Features: {X.shape[1]} columns")
print(f"Samples: {X.shape[0]} commits")
print(f"Target distribution: {y.value_counts().to_dict()}")

Features: 242 columns
Samples: 1910 commits
Target distribution: {0: 1426, 1: 484}


In [20]:
# Split into train (80%) and test (20%).
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} samples")
print(f"Test: {len(X_test)} samples")
print(f"Train fault rate: {y_train.mean():.1%}")
print(f"Test fault rate: {y_test.mean():.1%}")

Train: 1528 samples
Test: 382 samples
Train fault rate: 25.3%
Test fault rate: 25.4%


In [21]:
# Model 1: Logistic Regression (baseline).
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)
lr_prob = lr_model.predict_proba(X_test)[:, 1]

print("=== Logistic Regression ===")
print(f"F1 Score: {f1_score(y_test, lr_pred):.3f}")
print(f"AUC: {roc_auc_score(y_test, lr_prob):.3f}")
print(f"\nClassification Report:")
print(classification_report(y_test, lr_pred, target_names=["Safe", "Fault-Inducing"]))

=== Logistic Regression ===
F1 Score: 0.057
AUC: 0.607

Classification Report:
                precision    recall  f1-score   support

          Safe       0.75      0.98      0.85       285
Fault-Inducing       0.38      0.03      0.06        97

      accuracy                           0.74       382
     macro avg       0.56      0.51      0.45       382
  weighted avg       0.65      0.74      0.65       382



In [22]:
# Model 2: XGBoost (stronger model).
# scale_pos_weight handles class imbalance by giving more weight
# to the minority class (fault-inducing commits).
scale = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=scale,
    random_state=42,
    use_label_encoder=False,
    eval_metric="logloss",
)
xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

print("=== XGBoost ===")
print(f"F1 Score: {f1_score(y_test, xgb_pred):.3f}")
print(f"AUC: {roc_auc_score(y_test, xgb_prob):.3f}")
print(f"\nClassification Report:")
print(classification_report(y_test, xgb_pred, target_names=["Safe", "Fault-Inducing"]))

=== XGBoost ===
F1 Score: 0.444
AUC: 0.651

Classification Report:
                precision    recall  f1-score   support

          Safe       0.82      0.69      0.75       285
Fault-Inducing       0.37      0.55      0.44        97

      accuracy                           0.65       382
     macro avg       0.59      0.62      0.60       382
  weighted avg       0.70      0.65      0.67       382



In [23]:
# Side-by-side comparison.
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "XGBoost"],
    "F1 Score": [
        f1_score(y_test, lr_pred),
        f1_score(y_test, xgb_pred),
    ],
    "AUC": [
        roc_auc_score(y_test, lr_prob),
        roc_auc_score(y_test, xgb_prob),
    ],
})
print("=== Model Comparison ===")
comparison

=== Model Comparison ===


,Model,F1 Score,AUC
0,Logistic Regression,0.057143,0.606891
1,XGBoost,0.443515,0.651474


### What These Numbers Tell Us

**Logistic Regression** performed poorly (F1 = 0.06), meaning it almost never predicted a commit as fault-inducing. This happens when a simple linear model can't find a clear boundary between the two classes in the feature space. The AUC of 0.61 shows it has some ability to rank commits (better than random at 0.50), but not enough to make useful binary predictions.

**XGBoost** performed significantly better (F1 = 0.44, AUC = 0.65). The tree-based model can capture non-linear patterns that logistic regression misses. An F1 of 0.44 means it correctly identifies some fault-inducing commits while keeping false positives manageable. This is a realistic result for fault prediction on a single project with limited data (1,910 commits).

**Why aren't these numbers higher?** Three reasons:

1. **Single project, limited data.** We trained and tested on only 1,910 commits from one project. In the Example notebook, we train across multiple projects for more robust results.
2. **Noisy labels.** The SZZ algorithm that generated the fault-inducing labels is known to have imprecision. Some commits labeled as fault-inducing may not actually be the root cause.
3. **Feature granularity.** We're using project-level metrics per commit, not file-level or method-level metrics. The signal is diluted across the entire codebase snapshot.

Despite these limitations, the XGBoost AUC of 0.65 means the model does carry useful information: commits it ranks as high-risk are more likely to be fault-inducing than those it ranks as low-risk. This ranking ability is what our prioritization module (Module 4) uses to decide which debt items to fix first.

### Interpreting the Results

- **F1 Score** balances precision (of those we flagged as fault-inducing, how many actually were?) and recall (of all actual fault-inducing commits, how many did we catch?). Higher is better.
- **AUC** measures the model's ability to rank fault-inducing commits higher than safe ones. 0.5 means random guessing, 1.0 means perfect ranking.

These are results for a single project (`commons-io`). Performance varies across projects because different codebases have different coding styles, complexity patterns, and fault rates. In the Example notebook, we evaluate across multiple projects and report the variance.

**Note:** This is the same type of model that powers Module 3 (Impact Quantification) in our pipeline. In the Example notebook, we train on multiple projects and save the model for use by the remediation agent.

---
## Section 8: Debt Type Classification

In Section 7, we predicted whether a commit introduces faults. Here we tackle a different question: given a SonarQube issue, can we predict its type (BUG, CODE_SMELL, or VULNERABILITY) from the code metrics of the commit where it was created?

This is the core of Module 2 (Debt Classification). Being able to classify debt types automatically helps prioritize remediation: bugs need immediate attention, code smells can be scheduled, and vulnerabilities require security review.

In [24]:
# Load issues for commons-io and get the analysis mapping.
issues = get_sonar_issues(conn, "org.apache:commons-io")

# We need the issue TYPE as our target.
# Filter to the three main types.
issues_filtered = issues[
    issues["TYPE"].isin(["BUG", "CODE_SMELL", "VULNERABILITY"])
].copy()

print(f"Total issues: {len(issues_filtered)}")
print(f"\nDistribution:")
print(issues_filtered["TYPE"].value_counts().to_string())

INFO:ai_technical_debt_utils:Loaded 4000 SONAR_ISSUES rows for project org.apache:commons-io


Total issues: 4000

Distribution:
TYPE
CODE_SMELL       3888
BUG                96
VULNERABILITY      16


In [25]:
# For classification, we use issue-level features from the SONAR_ISSUES table.
# Available features: SEVERITY, EFFORT, DEBT, and the rule that triggered it.

# Convert SEVERITY to numeric.
severity_map = {
    "BLOCKER": 5, "CRITICAL": 4, "MAJOR": 3, "MINOR": 2, "INFO": 1
}
issues_filtered["SEVERITY_NUM"] = issues_filtered["SEVERITY"].map(severity_map)

# Convert EFFORT and DEBT to numeric (they may contain text like "30min").
issues_filtered["EFFORT_NUM"] = pd.to_numeric(
    issues_filtered["EFFORT"], errors="coerce"
).fillna(0)
issues_filtered["DEBT_NUM"] = pd.to_numeric(
    issues_filtered["DEBT"], errors="coerce"
).fillna(0)

# Use these as features.
feature_cols = ["SEVERITY_NUM", "EFFORT_NUM", "DEBT_NUM"]
X_issues = issues_filtered[feature_cols].fillna(0)
y_issues = issues_filtered["TYPE"]

print(f"Feature matrix: {X_issues.shape}")
print(f"Classes: {y_issues.unique().tolist()}")

Feature matrix: (4000, 3)
Classes: ['CODE_SMELL', 'BUG', 'VULNERABILITY']


In [26]:
from sklearn.preprocessing import LabelEncoder

# Train/test split.
X_tr, X_te, y_tr, y_te = train_test_split(
    X_issues, y_issues, test_size=0.2, random_state=42, stratify=y_issues
)

# XGBoost needs numeric labels.
le = LabelEncoder()
y_tr_encoded = le.fit_transform(y_tr)
y_te_encoded = le.transform(y_te)

# Train XGBoost multi-class classifier.
xgb_cls = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    use_label_encoder=False,
    eval_metric="mlogloss",
)
xgb_cls.fit(X_tr, y_tr_encoded)

y_pred_encoded = xgb_cls.predict(X_te)

# Convert predictions back to original labels for the report.
y_pred_labels = le.inverse_transform(y_pred_encoded)

print("=== Debt Type Classification (XGBoost) ===")
print(classification_report(y_te, y_pred_labels))

=== Debt Type Classification (XGBoost) ===
               precision    recall  f1-score   support

          BUG       0.80      0.84      0.82        19
   CODE_SMELL       0.99      1.00      0.99       778
VULNERABILITY       0.00      0.00      0.00         3

     accuracy                           0.99       800
    macro avg       0.60      0.61      0.61       800
 weighted avg       0.99      0.99      0.99       800



### Interpreting the Results

- **CODE_SMELL (F1 = 0.99):** Near-perfect classification because code smells are 97% of the data (778 out of 800 test samples). The model learns to predict the majority class easily.
- **BUG (F1 = 0.82):** Strong performance despite only 19 test samples. The model can distinguish bugs from code smells based on severity and effort features.
- **VULNERABILITY (F1 = 0.00):** Complete failure, but with only 3 test samples, this is expected. The model has almost no examples to learn from.

This reveals two important lessons:

1. **Class imbalance matters.** With 97% code smells, 2.4% bugs, and 0.4% vulnerabilities, the model is heavily biased toward predicting code smells. In the Example notebook, we address this with oversampling or class weighting.

2. **More features are needed.** We only used three features here (severity, effort, debt). Adding code metrics from `SONAR_MEASURES` (complexity, coverage, duplication) and rule-based features from the issue's `RULE` column would give the model much more signal to work with.

Despite these limitations, the fact that bugs are classifiable with F1 = 0.82 from just three features is promising. It suggests that with a richer feature set across multiple projects, a robust classifier is achievable.

---
## Section 9: Code Generation with a Pretrained Code Model

Our remediation agent needs a model that can understand and generate code. We use **Qwen2.5-Coder-0.5B-Instruct**, an open-source code model from Alibaba Cloud trained on 5.5 trillion tokens of source code (Hui et al., 2024).

Key properties of this model:
- **Size:** 0.5 billion parameters (~1 GB on disk)
- **Specialization:** Trained specifically for code tasks (generation, reasoning, fixing)
- **License:** Apache 2.0 (free for any use)
- **Runs on CPU:** No GPU required, though generation is slower

We chose this model because it balances quality and practicality: it's small enough to run on a laptop inside Docker, but capable enough to generate meaningful refactoring suggestions. For comparison, the ACE study (Tornhill et al., 2025) used GPT-4, which requires API access and costs money per request.

**Note:** The first time you run the cell below, it will download the model from Hugging Face (~1 GB). This takes a few minutes. Subsequent runs use the cached version.

In [27]:
!pip install accelerate -q

In [28]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load the code model and tokenizer.
model_name = "Qwen/Qwen2.5-Coder-0.5B-Instruct"

print(f"Loading model: {model_name}")
print("(First run downloads ~1 GB from Hugging Face...)\n")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
)

print("Model loaded successfully.")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-0.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-0.5B-Instruct/ea3f2471cf1b1f0db85067f1ef93848e38e88c25/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-0.5B-Instruct/ea3f2471cf1b1f0db85067f1ef93848e38e88c25/config.json "HTTP/1.1 200 OK"


Loading model: Qwen/Qwen2.5-Coder-0.5B-Instruct
(First run downloads ~1 GB from Hugging Face...)



config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-0.5B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-0.5B-Instruct/ea3f2471cf1b1f0db85067f1ef93848e38e88c25/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-0.5B-Instruct/ea3f2471cf1b1f0db85067f1ef93848e38e88c25/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-0.5B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-0.5B-Instruct/ea3f2471cf1b1f0db85067f1ef93848e38e88c25/generation_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-0.5B-Instruct/ea3f2471cf1b1f0db85067f1ef93848e38e88c25/generation_config.json "HTTP/1.1 200 OK"


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded successfully.
Parameters: 494,032,768


In [29]:
import torch

def generate_code_response(prompt, system_msg=None, max_tokens=256, seed=42):
    """
    Send a prompt to the code model and get a reproducible response.
    Setting the seed ensures the same output on every run.
    """
    if system_msg is None:
        system_msg = (
            "You are a code refactoring assistant. "
            "Given code with technical debt issues, suggest a cleaner version. "
            "Return only the improved code, no explanations."
        )
    
    # Set seed for reproducibility.
    torch.manual_seed(seed)
    
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_tokens,
        do_sample=False,  # Greedy decoding for deterministic output.
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]
    response = tokenizer.decode(output_ids, skip_special_tokens=True)
    return response

print("Helper function defined (with reproducibility settings).")

Helper function defined (with reproducibility settings).


In [30]:
import time

# Ask the model to refactor a messy function.
messy_function = '''
def calc(x, y, op):
    if op == 'a':
        r = x + y
    elif op == 's':
        r = x - y
    elif op == 'm':
        r = x * y
    elif op == 'd':
        if y != 0:
            r = x / y
        else:
            r = None
    else:
        r = None
    return r
'''

prompt = f"""The following Python function has technical debt issues:
- Single-letter variable names (poor readability)
- No docstring
- No type hints
- Could use a dictionary dispatch pattern instead of if/elif chain

Please refactor it:

{messy_function}"""

print("=== Original Code ===")
print(messy_function)

print("=== Model's Suggested Refactoring ===")
start_time = time.time()
refactored = generate_code_response(prompt)
elapsed = time.time() - start_time
print(refactored)
print(f"\n(Generated in {elapsed:.1f} seconds on CPU)")

=== Original Code ===

def calc(x, y, op):
    if op == 'a':
        r = x + y
    elif op == 's':
        r = x - y
    elif op == 'm':
        r = x * y
    elif op == 'd':
        if y != 0:
            r = x / y
        else:
            r = None
    else:
        r = None
    return r

=== Model's Suggested Refactoring ===
```python
def calc(x, y, op):
    # Define a dictionary to map operations to their respective functions
    ops = {
        'a': lambda x, y: x + y,
        's': lambda x, y: x - y,
        'm': lambda x, y: x * y,
        'd': lambda x, y: x / y if y != 0 else None
    }
    
    # Check if the operation is valid
    if op not in ops:
        return None
    
    # Get the function from the dictionary
    func = ops[op]
    
    # Call the function with the provided arguments
    result = func(x, y)
    
    return result
```

This refactored version uses a dictionary to map operations to their respective functions, making the code more readable and maintainabl

### What to Notice

The model produced a clean refactoring: it replaced the if/elif chain with a dictionary dispatch pattern, added input validation for unknown operations, and used descriptive variable names. Cyclomatic complexity would drop from 6 to 2 with this change.

However, note the generation time (~83 seconds on CPU). This is a practical constraint for our pipeline. If each debt item takes over a minute to process, running the agent on hundreds of items would take hours. This is why our pipeline prioritizes first (Modules 2-4) and only sends the top-k highest-impact items to the agent.

For comparison, the ACE study (Tornhill et al., 2025) used commercial API-based LLMs that respond in 1-2 seconds but cost money per request. Our approach trades speed for zero cost.

The model also claims it "includes type hints" but the output contains no type hints. This is a known behavior of small language models: they sometimes describe what they intended rather than what they actually produced. Our pipeline validates outputs using objective metrics, not the model's self-description.

---
## Section 10: Agent Building Blocks with LangGraph

The previous sections showed individual tools: metric computation, debt classification, impact prediction, and code generation. The remediation agent combines all of these into a multi-step workflow.

We use **LangGraph** to orchestrate this workflow. LangGraph models the agent as a directed graph where:
- **Nodes** are actions (compute metrics, generate fix, validate)
- **Edges** are transitions between actions
- **State** is a shared dictionary that accumulates information as the agent works

The agent follows this flow:
1. **Analyze:** Read the code and compute its current metrics
2. **Generate:** Ask the code model for a refactoring suggestion
3. **Validate:** Check if the suggestion is valid Python and actually improves the metrics
4. **Decide:** Accept the fix if metrics improved, reject it otherwise

In this section, we demonstrate the individual building blocks. The full agent loop runs in the Example notebook.

In [31]:
import ast as ast_module
import textwrap
from radon.complexity import cc_visit

# === Tool 1: Compute code metrics ===
def compute_metrics(code: str) -> dict:
    """
    Compute complexity metrics for a piece of Python code.
    Returns a dictionary with cyclomatic complexity, number of
    functions, and lines of code.
    """
    try:
        cc_results = cc_visit(code)
        total_cc = sum(item.complexity for item in cc_results)
        num_functions = len(cc_results)
    except Exception:
        total_cc = -1
        num_functions = 0
    
    loc = len([l for l in code.strip().splitlines() if l.strip()])
    
    return {
        "cyclomatic_complexity": total_cc,
        "num_functions": num_functions,
        "lines_of_code": loc,
    }

# === Tool 2: Validate Python syntax ===
def validate_syntax(code: str) -> dict:
    """
    Check if a string is valid Python code.
    Returns a dictionary with is_valid (bool) and error (str or None).
    """
    try:
        ast_module.parse(code)
        return {"is_valid": True, "error": None}
    except SyntaxError as e:
        return {"is_valid": False, "error": str(e)}

# === Tool 3: Compare before/after metrics ===
def compare_metrics(before: dict, after: dict) -> dict:
    """
    Compare metrics before and after a refactoring.
    Returns a dictionary indicating whether each metric improved.
    """
    return {
        "complexity_change": after["cyclomatic_complexity"] - before["cyclomatic_complexity"],
        "loc_change": after["lines_of_code"] - before["lines_of_code"],
        "complexity_improved": after["cyclomatic_complexity"] < before["cyclomatic_complexity"],
        "loc_improved": after["lines_of_code"] < before["lines_of_code"],
    }

# Test the tools on our messy function.
print("=== Metrics for messy function ===")
before_metrics = compute_metrics(messy_code)
print(before_metrics)

print("\n=== Syntax check ===")
print(validate_syntax(messy_code))

=== Metrics for messy function ===
{'cyclomatic_complexity': 16, 'num_functions': 1, 'lines_of_code': 32}

=== Syntax check ===
{'is_valid': True, 'error': None}


In [32]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

# Define the state that flows through the agent.
class AgentState(TypedDict):
    code: str
    metrics_before: dict
    suggestion: str
    metrics_after: dict
    is_valid: bool
    is_improved: bool
    attempt: int
    status: str

# === Node functions ===

def analyze_node(state: AgentState) -> dict:
    """Compute metrics on the original code."""
    metrics = compute_metrics(state["code"])
    return {
        "metrics_before": metrics,
        "attempt": 0,
        "status": f"Analyzed: CC={metrics['cyclomatic_complexity']}, LOC={metrics['lines_of_code']}",
    }

def generate_node(state: AgentState) -> dict:
    """Generate a refactoring suggestion with a specific, targeted prompt."""
    attempt = state.get("attempt", 0) + 1
    
    # Different strategies for each attempt.
    if attempt == 1:
        # First attempt: ask for a specific refactoring pattern.
        instruction = (
            "Refactor this Python function by:\n"
            "1. Extract the grade calculation (the if/elif chain for A/B/C/D/F) "
            "into a separate helper function\n"
            "2. Use a list of (threshold, grade) tuples instead of if/elif\n"
            "Return only the refactored Python code."
        )
    else:
        # Second attempt: even more specific.
        instruction = (
            "Split this single function into two functions:\n"
            "- A function called 'get_grade(score)' that takes a numeric score "
            "and returns a letter grade using a simple lookup\n"
            "- The main function that handles filtering and sorting, "
            "calling get_grade() for the grading step\n"
            "Return only the Python code."
        )
    
    prompt = f"{instruction}\n\n{state['code']}"
    suggestion = generate_code_response(prompt, max_tokens=400)
    
    # Extract code from markdown blocks if present.
    if "```python" in suggestion:
        suggestion = suggestion.split("```python")[1].split("```")[0]
    elif "```" in suggestion:
        suggestion = suggestion.split("```")[1].split("```")[0]
    
    return {
        "suggestion": suggestion.strip(),
        "attempt": attempt,
        "status": f"Generated suggestion (attempt {attempt})",
    }

def validate_node(state: AgentState) -> dict:
    """Check syntax and compute metrics on the suggestion."""
    syntax = validate_syntax(state["suggestion"])
    if not syntax["is_valid"]:
        return {
            "is_valid": False,
            "is_improved": False,
            "metrics_after": {},
            "status": f"Attempt {state['attempt']}: Invalid syntax - {syntax['error']}",
        }
    
    metrics_after = compute_metrics(state["suggestion"])
    comparison = compare_metrics(state["metrics_before"], metrics_after)
    
    return {
        "is_valid": True,
        "is_improved": comparison["complexity_improved"],
        "metrics_after": metrics_after,
        "status": (
            f"Attempt {state['attempt']}: CC {state['metrics_before']['cyclomatic_complexity']} -> "
            f"{metrics_after['cyclomatic_complexity']} "
            f"({'improved' if comparison['complexity_improved'] else 'not improved'})"
        ),
    }

def should_retry(state: AgentState) -> str:
    """Decide whether to retry or finish."""
    if state.get("is_improved"):
        return "done"
    if state.get("attempt", 0) >= 2:
        return "done"
    return "retry"

# === Build the graph with retry logic ===
builder = StateGraph(AgentState)
builder.add_node("analyze", analyze_node)
builder.add_node("generate", generate_node)
builder.add_node("validate", validate_node)

builder.add_edge(START, "analyze")
builder.add_edge("analyze", "generate")
builder.add_edge("generate", "validate")

# After validation, either retry or finish.
builder.add_conditional_edges(
    "validate",
    should_retry,
    {"retry": "generate", "done": END},
)

agent = builder.compile()

print("Agent graph compiled with retry logic.")
print("Flow: START -> analyze -> generate -> validate -> (retry or END)")

Agent graph compiled with retry logic.
Flow: START -> analyze -> generate -> validate -> (retry or END)


In [33]:
import time

print("=== Running Remediation Agent (with retry) ===\n")
start_time = time.time()

result = agent.invoke({"code": messy_code})

elapsed = time.time() - start_time

print(f"Status: {result['status']}")
print(f"Attempts: {result['attempt']}")
print(f"Valid Python: {result['is_valid']}")
print(f"Metrics improved: {result['is_improved']}")
print(f"Total time: {elapsed:.1f} seconds")

if result["metrics_before"] and result.get("metrics_after"):
    print(f"\n=== Before ===")
    print(f"  Cyclomatic Complexity: {result['metrics_before']['cyclomatic_complexity']}")
    print(f"  Lines of Code: {result['metrics_before']['lines_of_code']}")
    print(f"\n=== After ===")
    print(f"  Cyclomatic Complexity: {result['metrics_after'].get('cyclomatic_complexity', 'N/A')}")
    print(f"  Lines of Code: {result['metrics_after'].get('lines_of_code', 'N/A')}")

if result.get("suggestion"):
    print(f"\n=== Suggested Refactoring ===")
    print(result["suggestion"])

=== Running Remediation Agent (with retry) ===

Status: Attempt 1: CC 16 -> 10 (improved)
Attempts: 1
Valid Python: True
Metrics improved: True
Total time: 132.3 seconds

=== Before ===
  Cyclomatic Complexity: 16
  Lines of Code: 32

=== After ===
  Cyclomatic Complexity: 10
  Lines of Code: 36

=== Suggested Refactoring ===
def process_student_data(students, include_inactive=False, sort_by=None,
                         min_score=0, max_score=100, normalize=False):
    """Process student records with multiple filtering and sorting options."""
    def calculate_grade(score):
        if score < min_score or score > max_score:
            return None
        if normalize and max_score != min_score:
            score = (score - min_score) / (max_score - min_score) * 100
        if score >= 90:
            return "A"
        elif score >= 80:
            return "B"
        elif score >= 70:
            return "C"
        elif score >= 60:
            return "D"
        else:
            r

### What This Demonstrates

The agent reduced cyclomatic complexity from 16 to 10 (a 37.5% reduction) on its first attempt. The model correctly applied the "Extract Method" refactoring, pulling the grade calculation into a separate `calculate_grade` function.

The agent succeeded here because the prompt was **specific**: instead of vaguely asking to "reduce complexity," we told the model exactly what refactoring to apply ("extract the grade calculation into a helper function"). This demonstrates a core principle of our pipeline:

1. **Module 2 (Classification)** identifies the type of debt (e.g., "complex conditional logic")
2. **The agent uses this classification** to select a specific refactoring strategy (e.g., "extract the conditional block into a helper function")
3. **The validation step** confirms the improvement with objective metrics (CC dropped from 16 to 10)

**Current limitation:** The refactoring prompts in this demo are hardcoded (we manually wrote "extract the grade calculation into a helper function"). In the Example notebook, the agent generates prompts dynamically based on the debt classification from Module 2. For example, if the classifier labels an issue as "complex conditional," the agent automatically selects the "extract method" strategy without any manual prompt writing.

**Reproducibility:** All model outputs in this notebook use greedy decoding (`do_sample=False`) with a fixed seed (`torch.manual_seed(42)`), ensuring identical results on every run.